# 📊 Dashboard de Ventas — Multi-Sucursal

Herramienta interactiva para analizar ventas de tiendas de construcción.
Funciona con **cualquier número de sucursales**: agrega filas nuevas al CSV
(con una columna `Sede` distinta) y el dashboard las reconoce automáticamente.

**Cómo usar:** ejecuta todas las celdas (`Cell > Run All` o ▶▶). Al final aparecen
los controles — cambia la sucursal, categoría o rango de fechas y las gráficas y KPI se actualizan solas.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
import nbformat
from ipywidgets import interact, interactive_output, VBox, HBox, HTML as WHTML
from IPython.display import display, clear_output

pd.options.display.float_format = '{:,.2f}'.format

## 1. Cargar y preparar datos

Cambia `CSV_PATH` por la ruta de tu archivo cuando quieras usar datos actualizados.

In [2]:
CSV_PATH = "ventas.csv"

df = pd.read_csv(CSV_PATH)

# Limpieza y columnas calculadas
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df['Mes'] = df['Date'].dt.to_period('M').astype(str)
df['Margen'] = df['Net Unit Price'] - df['Unit Cost']
df['Margen_pct'] = (df['Margen'] / df['Net Unit Price']) * 100
df['Descuento_pct'] = ((df['Price List'] - df['Net Unit Price']) / df['Price List']) * 100

print(f"{len(df):,} líneas | {df['Invoice No'].nunique():,} facturas | "
      f"{df['Sede'].nunique()} sucursales: {list(df['Sede'].unique())}")
print(f"Rango de fechas: {df['Date'].min().date()} a {df['Date'].max().date()}")
df.head()

1,009 líneas | 400 facturas | 2 sucursales: ['DEL', 'PAL']
Rango de fechas: 2025-01-01 a 2026-06-27


,Sede,Invoice No,Date,Line,Product Code,Invoice Quantity,Unit Cost,Net Unit Price,Price List,Price Limit,Price Std,Category Name,Total,Mes,Margen,Margen_pct,Descuento_pct
0,DEL,DEL1000,2025-05-12,10,PLY-680,2,11.45,21.20,23.32,20.14,21.20,Plywood,42.40,2025-05,9.75,45.99,9.09
1,DEL,DEL1001,2025-08-17,10,CEM-425,1,23.54,34.87,38.36,33.13,34.87,Cemento,34.87,2025-08,11.33,32.49,9.10
2,DEL,DEL1001,2025-08-17,20,PLY-994,1,10.94,19.58,21.54,18.60,19.58,Plywood,19.58,2025-08,8.64,44.13,9.10
3,DEL,DEL1001,2025-08-17,30,TUB-134,2,33.42,58.77,64.65,55.83,58.77,Tuberías,117.54,2025-08,25.35,43.13,9.10
4,DEL,DEL1002,2026-06-23,10,PIN-721,3,22.66,32.27,35.50,30.66,32.27,Pinturas,96.81,2026-06,9.61,29.78,9.10


## 2. Controles interactivos

Estos widgets son la clave para que funcione con cualquier sucursal: el dropdown de `Sede` se llena solo a partir de los datos, así que si agregas una tienda nueva aparece automáticamente en la lista.

In [ ]:
sede_options = ['Todas'] + sorted(df['Sede'].unique().tolist())
cat_options = ['Todas'] + sorted(df['Category Name'].unique().tolist())

sede_w = widgets.Dropdown(options=sede_options, value='Todas', description='Sucursal:')
cat_w = widgets.Dropdown(options=cat_options, value='Todas', description='Categoría:')
fecha_w = widgets.SelectionRangeSlider(
    options=sorted(df['Mes'].unique()),
    index=(0, len(df['Mes'].unique())-1),
    description='Periodo:',
    layout=widgets.Layout(width='500px')
)

controles = HBox([sede_w, cat_w])
display(controles, fecha_w)

SelectionRangeSlider(description='Periodo:', index=(0, 17), layout=Layout(width='500px'), options=('2025-01', …

## 3. KPI y gráficas (se actualizan con los controles de arriba)

In [4]:
out = widgets.Output()

def filtrar(sede, categoria, periodo):
    d = df.copy()
    if sede != 'Todas':
        d = d[d['Sede'] == sede]
    if categoria != 'Todas':
        d = d[d['Category Name'] == categoria]
    d = d[(d['Mes'] >= periodo[0]) & (d['Mes'] <= periodo[1])]
    return d

def render(sede, categoria, periodo):
    with out:
        clear_output(wait=True)
        d = filtrar(sede, categoria, periodo)

        if d.empty:
            print("No hay datos para este filtro.")
            return

        # ---- KPI resumen ----
        ventas_totales = d['Total'].sum()
        n_facturas = d['Invoice No'].nunique()
        ticket_prom = d.groupby('Invoice No')['Total'].sum().mean()
        margen_pct_prom = d['Margen_pct'].mean()
        descuento_prom = d['Descuento_pct'].mean()
        unidades = d['Invoice Quantity'].sum()

        kpi_html = f'''
        <div style="display:flex; gap:16px; flex-wrap:wrap; margin-bottom:16px;">
          <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
            <div style="font-size:12px; color:#666;">VENTAS TOTALES</div>
            <div style="font-size:22px; font-weight:bold;">${ventas_totales:,.2f}</div>
          </div>
          <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
            <div style="font-size:12px; color:#666;">FACTURAS</div>
            <div style="font-size:22px; font-weight:bold;">{n_facturas:,}</div>
          </div>
          <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
            <div style="font-size:12px; color:#666;">TICKET PROMEDIO</div>
            <div style="font-size:22px; font-weight:bold;">${ticket_prom:,.2f}</div>
          </div>
          <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
            <div style="font-size:12px; color:#666;">MARGEN % PROMEDIO</div>
            <div style="font-size:22px; font-weight:bold;">{margen_pct_prom:,.1f}%</div>
          </div>
          <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
            <div style="font-size:12px; color:#666;">DESCUENTO PROMEDIO</div>
            <div style="font-size:22px; font-weight:bold;">{descuento_prom:,.1f}%</div>
          </div>
          <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
            <div style="font-size:12px; color:#666;">UNIDADES VENDIDAS</div>
            <div style="font-size:22px; font-weight:bold;">{unidades:,}</div>
          </div>
        </div>
        '''
        display(WHTML(kpi_html))

        # ---- Ventas por mes (y por sede si aplica) ----
        ventas_mes = d.groupby(['Mes', 'Sede'])['Total'].sum().reset_index()
        fig1 = px.line(ventas_mes, x='Mes', y='Total', color='Sede', markers=True,
                        title='Ventas por mes')
        fig1.update_layout(height=350, margin=dict(t=40, b=20))
        fig1.show()

        # ---- Ventas y margen por categoría ----
        cat_summary = d.groupby('Category Name').agg(
            Ventas=('Total', 'sum'),
            Margen_pct=('Margen_pct', 'mean')
        ).reset_index().sort_values('Ventas', ascending=False)

        fig2 = go.Figure()
        fig2.add_bar(x=cat_summary['Category Name'], y=cat_summary['Ventas'], name='Ventas ($)')
        fig2.add_scatter(x=cat_summary['Category Name'], y=cat_summary['Margen_pct'],
                          name='Margen %', yaxis='y2', mode='lines+markers', line=dict(color='crimson'))
        fig2.update_layout(
            title='Ventas por categoría (barras) vs. Margen % promedio (línea)',
            yaxis=dict(title='Ventas ($)'),
            yaxis2=dict(title='Margen %', overlaying='y', side='right'),
            height=380, margin=dict(t=40, b=20)
        )
        fig2.show()

        # ---- Comparativo entre sucursales (si hay más de una en el filtro) ----
        if d['Sede'].nunique() > 1:
            sede_summary = d.groupby('Sede').agg(
                Ventas=('Total', 'sum'),
                Ticket_prom=('Total', lambda x: d.loc[x.index].groupby('Invoice No')['Total'].sum().mean()),
                Margen_pct=('Margen_pct', 'mean')
            ).reset_index()
            fig3 = px.bar(sede_summary, x='Sede', y='Ventas', color='Sede',
                           title='Comparativo de ventas por sucursal', text_auto='.2s')
            fig3.update_layout(height=320, margin=dict(t=40, b=20), showlegend=False)
            fig3.show()

interactive_output(render, {'sede': sede_w, 'categoria': cat_w, 'periodo': fecha_w})
display(out)

Output()

## 4. Cómo escalar esto a más sucursales

- **Agregar una tienda nueva:** solo pega sus filas en el CSV con su propio código de `Sede` (por ejemplo `GYE`). No hay que tocar el código — los dropdowns se generan a partir de los datos.
- **Automatizar la carga:** si cada sucursal exporta su propio archivo, puedes concatenarlos con `pd.concat([pd.read_csv(f) for f in lista_de_archivos])` antes de la celda de limpieza.
- **Compartirlo como app (sin mostrar código):** instala `voila` (`pip install voila`) y corre `voila este_notebook.ipynb` — abre una página web con solo los controles y las gráficas, ideal para que lo use alguien que no usa Jupyter.
- **Actualización periódica:** si el CSV se sobrescribe cada semana/mes con datos nuevos, basta con volver a correr `Run All` para refrescar todo el dashboard.